# 🖥️ TOCAIA OSINT Linux — Desktop no Colab (ver a cara do sistema, sem instalar nada)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ridd1kulusC0d3r/TOCAIALinux/blob/main/nuvem/colab/TOCAIA_Colab_Desktop.ipynb)

O **desktop Xfce da TOCAIA** — wallpaper da onça, menu por fases, terminal com o `trem` —
rodando **na máquina do Colab** e aberto **no seu navegador** por um link. Sem instalar nada.

> **Honesto:** não emulamos o boot da ISO (QEMU sem KVM seria lento demais). Montamos **o
> mesmo desktop** na VM do Colab e mostramos por **Xvfb + x11vnc + noVNC + túnel Cloudflare**.
> É demo: vídeo por VNC é lento, o Colab derruba sessão ociosa, e o link é público/temporário.

Para **usar** OSINT de leve, prefira o notebook otimizado `TOCAIA_OSINT_Linux.ipynb`.

**Passos:** *Ambiente de execução → Executar tudo* e, ao fim, clique no link `…trycloudflare.com/vnc.html`.


## 1 · Baixar a TOCAIA e gerar o tema (onça, menu, ícones)


In [ ]:
import os, subprocess
REPO='https://github.com/Ridd1kulusC0d3r/TOCAIALinux'; BRANCH='main'
if not os.path.exists('/content/TOCAIALinux'):
    subprocess.run(['git','clone','--depth','1','-b',BRANCH,REPO,'/content/TOCAIALinux'], check=True)
os.chdir('/content/TOCAIALinux')
subprocess.run(['python3','assets/gerar_marca.py','--destino','assets/gerado'], check=True)
subprocess.run(['python3','tema/gerar_tema.py','--destino','/'], check=True)
subprocess.run(['bash','scripts/empacotar.sh'], check=True)
subprocess.run('install -d /opt/tocaia/ferramentas', shell=True, check=True)
subprocess.run('cp catalogo/catalogo.json /opt/tocaia/ && cp ferramentas/*.pyz /opt/tocaia/ferramentas/ && cp -r casos wargames /opt/tocaia/ 2>/dev/null; true', shell=True)
subprocess.run('install -m0755 trem/trem.py /usr/local/bin/trem && ln -sf /usr/local/bin/trem /usr/local/bin/tocaia', shell=True, check=True)
print('TOCAIA preparada.')


## 2 · Instalar desktop + Xvfb/x11vnc + noVNC + túnel
Sem TigerVNC (o `vncserver` dá erro em headless). Usamos **Xvfb** (tela virtual) + **x11vnc**. (~2–3 min)


In [ ]:
%%bash
set -e
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
apt-get install -y -qq xfce4 xfce4-terminal xfce4-whiskermenu-plugin dbus-x11 \
   xvfb x11vnc novnc websockify librsvg2-bin fonts-dejavu-core >/dev/null 2>&1 || \
   apt-get install -y -qq xfce4 xfce4-terminal dbus-x11 xvfb x11vnc novnc websockify librsvg2-bin
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared
[ -f /usr/share/tocaia/wallpaper.svg ] && rsvg-convert -w 1600 -h 900 /usr/share/tocaia/wallpaper.svg -o /usr/share/tocaia/wallpaper.png || true
mkdir -p /root/.config && cp -r /etc/skel/.config/* /root/.config/ 2>/dev/null || true
echo 'desktop instalado.'


## 3 · Subir a tela virtual e pegar o link
Inicia Xvfb → Xfce → x11vnc → noVNC → túnel. **Clique no link** `…trycloudflare.com/vnc.html` que aparecer.


In [ ]:
import subprocess, time, re, os
os.environ['DISPLAY']=':1'
# limpa sessões anteriores
subprocess.run('pkill -f Xvfb; pkill -f x11vnc; pkill -f startxfce4; pkill -f websockify; pkill -f cloudflared; true', shell=True)
time.sleep(2)
# 1) tela virtual
subprocess.Popen('Xvfb :1 -screen 0 1600x900x24 -ac +extension GLX +render -noreset >/content/xvfb.log 2>&1', shell=True)
time.sleep(3)
# 2) sessão Xfce (com dbus)
subprocess.Popen('export DISPLAY=:1; dbus-launch --exit-with-session startxfce4 >/content/xfce.log 2>&1', shell=True)
time.sleep(8)
# 3) wallpaper da onça em todos os monitores da sessão
subprocess.run('''export DISPLAY=:1
ps=$(xfconf-query -c xfce4-desktop -l 2>/dev/null | grep last-image || true)
if [ -z "$ps" ]; then
  xfconf-query -c xfce4-desktop -p /backdrop/screen0/monitorscreen/workspace0/last-image -n -t string -s /usr/share/tocaia/wallpaper.png 2>/dev/null || true
else
  for p in $ps; do xfconf-query -c xfce4-desktop -p "$p" -s /usr/share/tocaia/wallpaper.png 2>/dev/null || true; done
fi
xfdesktop --reload 2>/dev/null || true''', shell=True)
# 4) x11vnc expõe a tela :1 (sem senha, só via túnel)
subprocess.Popen('x11vnc -display :1 -nopw -forever -shared -rfbport 5901 -quiet >/content/x11vnc.log 2>&1', shell=True)
time.sleep(3)
# 5) noVNC (6080) -> VNC (5901)
subprocess.Popen('websockify --web=/usr/share/novnc 6080 localhost:5901 >/content/novnc.log 2>&1', shell=True)
time.sleep(3)
# 6) túnel Cloudflare
proc=subprocess.Popen('cloudflared tunnel --url http://localhost:6080 --no-autoupdate',
                      shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url=None
for _ in range(90):
    line=proc.stdout.readline()
    if not line: time.sleep(0.5); continue
    m=re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m: url=m.group(0); break
print('\n'+'='*60)
if url:
    print('✅ Abra a área de trabalho da TOCAIA aqui:')
    print('   '+url+'/vnc.html?autoconnect=1&resize=remote')
else:
    print('Túnel não subiu. Veja os logs em /content/*.log e rode esta célula de novo.')
print('='*60)


## Se algo falhar
Rode a célula abaixo para ver os logs (Xvfb, Xfce, x11vnc, noVNC) e me mande a saída.


In [ ]:
for f in ['xvfb','xfce','x11vnc','novnc']:
    print('\n===== /content/%s.log ====='%f)
    try: print(open('/content/%s.log'%f).read()[-1500:])
    except FileNotFoundError: print('(sem log)')


## Como usar
- No link, você vê a **área de trabalho da TOCAIA** (wallpaper da onça).
- Menu **TOCAIA** → ferramentas por **fase**. Abra o **Terminal** e rode `trem`, `trem relogio --demo`.

Demo: VNC é lento; o Colab encerra sessão ociosa; link público/temporário — não deixe dado sensível na tela.

---
**DEIVOSINT · OSINT no Jutsu** · Fonte pública · zero interação · base legal.
